# 00. 베이스라인 + 공통 교차검증

- 담당: **① 장성은 (팀장)**
- 팀 공통 기준 점수(CV Macro F1)예요. 모든 실험은 이 점수와 비교해요.
- 피처: 팀 기본 피처 `binarize_mutations` (WT·결측 = 0, 변이 = 1)
- 모델: XGBoost (운영진 베이스라인과 비슷한 설정)
- 누수 방지: 전처리와 모델을 Pipeline으로 묶어 fold의 학습 부분에만 fit. test는 형식 확인과 최종 추론에만.

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-develop) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: C:\Users\user\Desktop\hakerton | DATA_DIR: C:\Users\user\Desktop\hakerton\data


In [2]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result, save_predictions
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: Windows-11-10.0.26300-SP0 | python 3.13.17 | pandas 3.1.1 | scikit-learn 1.9.1


In [3]:
import xgboost as xgb
print('xgboost', xgb.__version__)

xgboost 3.4.1


## 1. 데이터

In [4]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


## 2. 모델 (Pipeline)

In [5]:
XGB_PARAMS = dict(n_estimators=200, learning_rate=0.1, max_depth=6,
                  tree_method="hist", random_state=SEED, n_jobs=-1)

def make_pipeline():
    return Pipeline([
        ("features", basic_transformer()),
        ("model", xgb.XGBClassifier(**XGB_PARAMS)),
    ])

## 3. 공통 교차검증 (5-fold, 로컬 기준 약 10분)

In [6]:
cv = run_cv(make_pipeline(), X, y, folds=folds)

fold 0: Macro F1 = 0.3200


fold 1: Macro F1 = 0.3194


fold 2: Macro F1 = 0.3412


fold 3: Macro F1 = 0.3089


fold 4: Macro F1 = 0.3146
CV Macro F1 = 0.3209 (std 0.0110)


## 4. 클래스별 F1
Macro F1은 26개 클래스를 같은 비중으로 평균해요. 낮은 클래스가 개선 포인트예요.

In [7]:
per_class_f1(y, cv['oof'], le.classes_).round(3)

,f1-score,support
THYM,0.000,98.0
KIRC,0.087,334.0
CESC,0.092,155.0
PAAD,0.131,120.0
PCPG,0.139,147.0
SARC,0.174,198.0
PRAD,0.187,266.0
KIPAN,0.196,515.0
LUSC,0.212,178.0
LUAD,0.213,184.0


## 5. 기록

In [8]:
log_result(owner="J_leader", notebook="00_baseline.ipynb", experiment="baseline_xgb",
           features="basic(WT=0,변이=1)", model="XGBoost", params=XGB_PARAMS, cv=cv,
           notes="운영진 베이스라인 설정 + 공통 CV")

기록: log_J_leader.csv ← baseline_xgb (CV 0.3209)


## 6. 전체 train 학습 → test 예측 → 제출 파일
제출은 팀장이 하루 4회 안에서 해요.

In [9]:
final_model = make_pipeline().fit(X, y)
pred = le.inverse_transform(final_model.predict(test.drop(columns=["ID"])))
sub = make_submission(test, pred, "00_baseline_xgb.csv")
sub.head()

저장: C:\Users\user\Desktop\hakerton\submissions\00_baseline_xgb.csv (2546행)


,ID,SUBCLASS
0,TEST_0000,STES
1,TEST_0001,UCEC
2,TEST_0002,THCA
3,TEST_0003,LGG
4,TEST_0004,HNSC


## 7. 앙상블 후보로 예측 저장
`experiments/preds/`에 저장해서 05_ensemble에서 섞을 수 있게 해요.

In [10]:
test_proba = final_model.predict_proba(test.drop(columns=["ID"]))
save_predictions(owner="J_leader", name="baseline_xgb", oof_proba=cv["oof_proba"],
                 test_proba=test_proba, classes=le.classes_, cv=cv, notes="팀 기준 모델")

예측 저장: J_leader__baseline_xgb (CV 0.3209) → experiments/preds/
